# 1. Data Load

In [340]:
import os
from pathlib import Path

In [341]:
data_path = Path("../data")

movie_path = f"{data_path}/movies.csv"
rating_path = f"{data_path}/ratings.csv"
tag_path = f"{data_path}/tags.csv"

In [342]:
import pandas as pd

movies_df = pd.read_csv(movie_path)
ratings_df = pd.read_csv(rating_path)
tags_df = pd.read_csv(tag_path)


## a) Null check

In [343]:
df_list = [movies_df, ratings_df, tags_df]

for df in df_list:
    print(df.isnull().sum(), '\n')

movieId    0
title      0
genres     0
dtype: int64 

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64 

userId       0
movieId      0
tag          0
timestamp    0
dtype: int64 



# b) index check

In [344]:
movies_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  9742 non-null   int64 
 1   title    9742 non-null   object
 2   genres   9742 non-null   object
dtypes: int64(1), object(2)
memory usage: 228.5+ KB


In [345]:
ratings_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


In [346]:
tags_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3683 entries, 0 to 3682
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   userId     3683 non-null   int64 
 1   movieId    3683 non-null   int64 
 2   tag        3683 non-null   object
 3   timestamp  3683 non-null   int64 
dtypes: int64(3), object(1)
memory usage: 115.2+ KB


movie id, user id 는 속도를 위해 int 형으로 그대로 사용했습니다.

# c) check duplicate

In [347]:
for col in movies_df:
    print(f"{col} : \t", movies_df[col].nunique())

movieId : 	 9742
title : 	 9737
genres : 	 951


In [348]:
duplicated_title = movies_df[movies_df['title'].duplicated()]['title']
movies_df[movies_df['title'].duplicated()]

,movieId,title,genres
5601,26958,Emma (1996),Romance
6932,64997,War of the Worlds (2005),Action|Sci-Fi
9106,144606,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Romance|Thriller
9135,147002,Eros (2004),Drama|Romance
9468,168358,Saturn 3 (1980),Sci-Fi|Thriller


In [349]:
dup_temp = movies_df[movies_df['title'].isin(duplicated_title)].sort_values(by='title')
dup_temp

,movieId,title,genres
4169,6003,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Thriller
9106,144606,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Romance|Thriller
650,838,Emma (1996),Comedy|Drama|Romance
5601,26958,Emma (1996),Romance
5854,32600,Eros (2004),Drama
9135,147002,Eros (2004),Drama|Romance
2141,2851,Saturn 3 (1980),Adventure|Sci-Fi|Thriller
9468,168358,Saturn 3 (1980),Sci-Fi|Thriller
5931,34048,War of the Worlds (2005),Action|Adventure|Sci-Fi|Thriller
6932,64997,War of the Worlds (2005),Action|Sci-Fi


In [350]:
# genres 가 가장 긴 영화 기준으로 중복 데이터 처리
dup_temp['genres_len'] = dup_temp['genres'].str.len()
dup_temp

,movieId,title,genres,genres_len
4169,6003,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Thriller,27
9106,144606,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Romance|Thriller,35
650,838,Emma (1996),Comedy|Drama|Romance,20
5601,26958,Emma (1996),Romance,7
5854,32600,Eros (2004),Drama,5
9135,147002,Eros (2004),Drama|Romance,13
2141,2851,Saturn 3 (1980),Adventure|Sci-Fi|Thriller,25
9468,168358,Saturn 3 (1980),Sci-Fi|Thriller,15
5931,34048,War of the Worlds (2005),Action|Adventure|Sci-Fi|Thriller,32
6932,64997,War of the Worlds (2005),Action|Sci-Fi,13


In [351]:
group_temp = dup_temp.groupby(by='title')
dup_title_max_len_idx = group_temp['genres_len'].idxmax() # 데이터 검산을 위한 drop idx 와 max_len_idx

movies_df.loc[dup_title_max_len_idx]

,movieId,title,genres
9106,144606,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Romance|Thriller
650,838,Emma (1996),Comedy|Drama|Romance
9135,147002,Eros (2004),Drama|Romance
2141,2851,Saturn 3 (1980),Adventure|Sci-Fi|Thriller
5931,34048,War of the Worlds (2005),Action|Adventure|Sci-Fi|Thriller


In [352]:
drop_idx = dup_temp.index.difference(dup_title_max_len_idx)
dup_temp.loc[drop_idx]

,movieId,title,genres,genres_len
4169,6003,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Thriller,27
5601,26958,Emma (1996),Romance,7
5854,32600,Eros (2004),Drama,5
6932,64997,War of the Worlds (2005),Action|Sci-Fi,13
9468,168358,Saturn 3 (1980),Sci-Fi|Thriller,15


In [353]:
drop_movieId = dup_temp.loc[drop_idx]['movieId']
max_len_movieId = dup_temp.loc[dup_title_max_len_idx]['movieId']

In [354]:
movies_refine = movies_df.drop(drop_idx).copy()

In [357]:
print("삭제된 ROW :\t", abs(len(movies_df) - len(movies_refine)))
print("dup index 갯수 :\t", len(drop_idx))

삭제된 ROW :	 5
dup index 갯수 :	 5


In [358]:
movies_refine.loc[dup_title_max_len_idx]

,movieId,title,genres
9106,144606,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Romance|Thriller
650,838,Emma (1996),Comedy|Drama|Romance
9135,147002,Eros (2004),Drama|Romance
2141,2851,Saturn 3 (1980),Adventure|Sci-Fi|Thriller
5931,34048,War of the Worlds (2005),Action|Adventure|Sci-Fi|Thriller


In [359]:
max_len_movieId

9106    144606
650        838
9135    147002
2141      2851
5931     34048
Name: movieId, dtype: int64

In [363]:
# 중복 제거 확인
movies_refine[movies_refine['movieId'].isin(max_len_movieId.values)]

,movieId,title,genres
650,838,Emma (1996),Comedy|Drama|Romance
2141,2851,Saturn 3 (1980),Adventure|Sci-Fi|Thriller
5931,34048,War of the Worlds (2005),Action|Adventure|Sci-Fi|Thriller
9106,144606,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Romance|Thriller
9135,147002,Eros (2004),Drama|Romance


In [364]:
movies_refine[movies_refine['movieId'].isin(drop_movieId.values)]

,movieId,title,genres


In [365]:
max_len_movieId.values

array([144606,    838, 147002,   2851,  34048])

In [366]:
idx_list = group_temp['movieId'].apply(list)
idx_list

title
Confessions of a Dangerous Mind (2002)     [6003, 144606]
Emma (1996)                                  [838, 26958]
Eros (2004)                               [32600, 147002]
Saturn 3 (1980)                            [2851, 168358]
War of the Worlds (2005)                   [34048, 64997]
Name: movieId, dtype: object

In [368]:
# mapping 을 위한 series 생성
map_series = dup_temp.loc[dup_title_max_len_idx].set_index('title')['movieId']
map_series

title
Confessions of a Dangerous Mind (2002)    144606
Emma (1996)                                  838
Eros (2004)                               147002
Saturn 3 (1980)                             2851
War of the Worlds (2005)                   34048
Name: movieId, dtype: int64

In [369]:
mapping_dict = {}
for title, movie_ids in idx_list.items():
    dict_value = map_series[title]
    for id in movie_ids:
        mapping_dict[id] = dict_value

In [370]:
mapping_dict

{6003: 144606,
 144606: 144606,
 838: 838,
 26958: 838,
 32600: 147002,
 147002: 147002,
 2851: 2851,
 168358: 2851,
 34048: 34048,
 64997: 34048}

In [372]:
# 중복치 제거 확인용
ratings_df[ratings_df['movieId'].isin(drop_movieId)]

,userId,movieId,rating,timestamp
4747,28,64997,3.5,1234850075
11451,68,64997,2.5,1230497715
17449,111,6003,4.0,1516468531
23053,156,6003,3.5,1106882187
26958,182,6003,3.0,1054780821
42984,288,6003,4.0,1066059244
54020,356,6003,4.5,1229139513
59953,387,6003,3.5,1208707060
64063,414,6003,3.5,1092414917
74530,474,6003,3.5,1087831997


In [373]:
ratings_refine = ratings_df['movieId'].map(mapping_dict)

In [374]:
ratings_refine = ratings_refine.fillna(ratings_df['movieId'])

In [377]:
# 정제 데이터를 기존 데이터로 변경
ratings_df['movieId'] = ratings_refine

In [ ]:
# 중복 제거 확인2
ratings_df[ratings_df['movieId'].isin(drop_movieId)]

,userId,movieId,rating,timestamp
